In [2]:
import os
import json
import requests
import time
import concurrent.futures
from typing import List, Dict, Set, Tuple
from collections import deque

from dotenv import load_dotenv

load_dotenv()

steamApiKey = os.getenv("STEAM_API_KEY")
startingSteamId = os.getenv("STEAM_ID")

if not steamApiKey:
    print("API ключ не найден в .env")
    exit()
elif not startingSteamId:
    print("STEAM_ID не найден в .env")
    exit()

### Сбор данных профиля

In [3]:
def loadCrawlQueue() -> List[Tuple[str, int]]:
    queueFile = "../data/steamCrawlQueue.json"
    if os.path.exists(queueFile):
        try:
            with open(queueFile, "r", encoding="utf-8") as f:
                return json.load(f)  # список [ [steamId, depth], ... ]
        except Exception:
            pass
    return []


def saveCrawlQueue(queue: List[Tuple[str, int]]) -> None:
    queueFile = "../data/steamCrawlQueue.json"
    os.makedirs(os.path.dirname(queueFile), exist_ok=True)
    with open(queueFile, "w", encoding="utf-8") as f:
        json.dump(queue, f, ensure_ascii=False, indent=2)


def continueCrawlForNewIds(
    steamApiKey: str,
    startingUserId: str,
    existingIds: Set[str],
    neededNew: int,
    maxDepth: int = 2
) -> List[str]:
    newIds: List[str] = []
    visitedForCrawl = set()
    queue: List[Tuple[str, int]] = loadCrawlQueue()

    if not queue:
        queue = [(startingUserId, 0)]
        print("Начало краулинга с начального пользователя...")

    while queue and len(newIds) < neededNew:
        currentUserId, currentDepth = queue.pop(0)

        if currentUserId in visitedForCrawl:
            continue
        visitedForCrawl.add(currentUserId)

        if currentUserId not in existingIds and currentUserId not in newIds:
            newIds.append(currentUserId)
            print(f"Найден новый пользователь {len(newIds)}/{neededNew}: {currentUserId} (глубина {currentDepth})...")

        if currentDepth < maxDepth:
            friendsIds = getFriendsListForUser(steamApiKey, currentUserId)
            for friendId in friendsIds[:200]:
                if friendId not in visitedForCrawl:
                    queue.append((friendId, currentDepth + 1))

        time.sleep(0.0)

    saveCrawlQueue(queue)
    return newIds


def fetchOwnedGamesForUser(steamApiKey: str, userSteamId: str) -> dict:
    url = "http://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        "key": steamApiKey,
        "steamid": userSteamId,
        "include_appinfo": True,
        "include_played_free_games": True,
        "format": "json"
    }
    maxRetries = 5
    for attempt in range(maxRetries):
        try:
            response = requests.get(url, params=params)

            if response.status_code in (420, 429):
                waitSeconds = 2 ** attempt + 1
                print(f"  Rate-limit ({response.status_code}) для {userSteamId}. Ждём {waitSeconds} сек... (попытка {attempt+1}/{maxRetries})")
                time.sleep(waitSeconds)
                continue

            if response.status_code != 200:
                return {"error": f"HTTP {response.status_code} (вероятно приватный профиль или ошибка API)"}
            
            data = response.json()
            responseData = data.get("response", {})

            if "games" not in responseData:
                return {"error": "Нет доступа к играм (профиль приватный или ошибка API)"}
            
            slimGames = []
            for game in responseData["games"]:
                slimGames.append({
                    "appid": game.get("appid"),
                    "name": game.get("name"),
                    "playtimeForever": game.get("playtime_forever")
                })
            
            return {
                "games": slimGames,
                "gameCount": responseData.get("game_count", 0)
            }
        except requests.exceptions.RequestException as e:
            return {"error": f"Сетевая ошибка: {str(e)}"}
        except json.JSONDecodeError:
            return {"error": "Пустой или некорректный JSON от Steam (rate-limit или временная ошибка)"}
        except Exception as e:
            return {"error": f"Неизвестная ошибка: {str(e)}"}


def fetchPlayerSummariesBatch(steamApiKey: str, userSteamIds: List[str]) -> Dict[str, dict]:
    if not userSteamIds:
        return {}
    
    results: Dict[str, dict] = {}
    batchSize = 100
    
    for i in range(0, len(userSteamIds), batchSize):
        batchIds = userSteamIds[i:i + batchSize]
        url = "http://api.steampowered.com/ISteamUser/GetPlayerSummaries/v0002/"
        params = {
            "key": steamApiKey,
            "steamids": ",".join(batchIds),
            "format": "json"
        }
        response = requests.get(url, params=params)
        data = response.json()
        players = data.get("response", {}).get("players", [])
        
        for player in players:
            steamId = player.get("steamid")
            if steamId:
                results[steamId] = {
                    "personaname": player.get("personaname"),
                    "communityvisibilitystate": player.get("communityvisibilitystate"),
                    "timecreated": player.get("timecreated"),
                    "loccountrycode": player.get("loccountrycode")
                }
    
    return results


def collectNewUserIds(
    steamApiKey: str,
    startingUserId: str,
    existingIds: Set[str],
    neededNew: int,
    maxDepth: int = 2
) -> List[str]:
    newIds: List[str] = []
    visitedForCrawl = set()
    queue = deque([(startingUserId, 0)])

    while queue and len(newIds) < neededNew:
        currentUserId, currentDepth = queue.popleft()
        
        if currentUserId in visitedForCrawl:
            continue
        visitedForCrawl.add(currentUserId)

        if currentUserId not in existingIds:
            newIds.append(currentUserId)
            print(f"Найден новый пользователь {len(newIds)}/{neededNew}: {currentUserId} (глубина {currentDepth})...")

        if currentDepth < maxDepth:
            friendsIds = getFriendsListForUser(steamApiKey, currentUserId)
            for friendId in friendsIds[:200]:  # лимит на ветвление
                if friendId not in visitedForCrawl:
                    queue.append((friendId, currentDepth + 1))
        
        time.sleep(0.0)

    return newIds


def fetchOwnedGamesForUserIds(
    steamApiKey: str,
    userSteamIds: List[str],
    maxWorkers: int = 5
) -> list:
    if not userSteamIds:
        return []
    
    print(f"Запускаем параллельный сбор OwnedGames для {len(userSteamIds)} пользователей "
          f"в {maxWorkers} потоках ...")
    
    def processSingleUser(userId: str) -> dict:
        ownedGamesResult = fetchOwnedGamesForUser(steamApiKey, userId)
        userEntry = {
            "steamId": userId,
            "gameCount": ownedGamesResult.get("gameCount", 0) if "gameCount" in ownedGamesResult else 0,
            "ownedGames": ownedGamesResult.get("games", []) if "games" in ownedGamesResult else [],
            "error": ownedGamesResult.get("error") if "error" in ownedGamesResult else None
        }
        time.sleep(0.3)
        return userEntry
    
    newEntries = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = [executor.submit(processSingleUser, uid) for uid in userSteamIds]
        
        for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
            try:
                userEntry = future.result()
                newEntries.append(userEntry)
                print(f"  [{i}/{len(userSteamIds)}] Завершён пользователь {userEntry['steamId']}")
            except Exception as e:
                print(f"  Критическая ошибка в потоке: {e}")
    
    return newEntries


def collectAccountData(
    steamApiKey: str,
    startingUserId: str,
    maxUsers: int = 30,
    maxDepth: int = 2,
    maxWorkersForOwnedGames: int = 5,
    datasetFilePath: str = "../data/steamUsersDataset.json"
) -> list:
    existingDataset: list = []
    existingIds: Set[str] = set()
    if os.path.exists(datasetFilePath):
        try:
            with open(datasetFilePath, "r", encoding="utf-8") as file:
                existingDataset = json.load(file)
            existingIds = {user.get("steamId") for user in existingDataset if user.get("steamId")}
            print(f"Найдено {len(existingDataset)} существующих профилей в {datasetFilePath}")
        except Exception as e:
            print(f"Ошибка чтения существующего файла: {e}. Начинаем с нуля.")
    
    currentCount = len(existingDataset)
    neededNew = max(0, maxUsers - currentCount)
    
    if neededNew == 0:
        print(f"Уже достаточно данных ({currentCount} >= {maxUsers}).")
        return existingDataset
    
    print(f"Нужно собрать ещё {neededNew} новых пользователей...")
    newUserIds = continueCrawlForNewIds(
        steamApiKey=steamApiKey,
        startingUserId=startingUserId,
        existingIds=existingIds,
        neededNew=neededNew,
        maxDepth=maxDepth
    )
    
    if not newUserIds:
        print("Не удалось найти новые ID.")
        return existingDataset
    
    print(f"Начинаем параллельный сбор OwnedGames для {len(newUserIds)} новых пользователей..."
          f"({maxWorkersForOwnedGames} потоков)...")
    newEntries = fetchOwnedGamesForUserIds(
        steamApiKey=steamApiKey,
        userSteamIds=newUserIds,
        maxWorkers=maxWorkersForOwnedGames
    )
    
    print("Получаем PlayerSummaries батчем для всех новых пользователей...")
    batchSummaries = fetchPlayerSummariesBatch(steamApiKey, newUserIds)
    
    for entry in newEntries:
        summary = batchSummaries.get(entry["steamId"], {})
        entry.update({
            "personaname": summary.get("personaname"),
            "communityvisibilitystate": summary.get("communityvisibilitystate"),
            "timecreated": summary.get("timecreated"),
            "loccountrycode": summary.get("loccountrycode")
        })
    
    fullDataset = existingDataset + newEntries
    print(f"Всего пользователей в датасете: {len(fullDataset)} (из них новых: {len(newEntries)})")
    
    return fullDataset

def getFriendsListForUser(steamApiKey: str, userSteamId: str) -> list:
    url = "http://api.steampowered.com/ISteamUser/GetFriendList/v0001/"
    params = {
        "key": steamApiKey,
        "steamid": userSteamId,
        "relationship": "friend",
        "format": "json"
    }

    maxRetries = 5
    for attempt in range(maxRetries):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code in (420, 429):
                waitSeconds = 2 ** attempt + 1
                print(f"  Rate-limit ({response.status_code}) при получении друзей {userSteamId}. Ждём {waitSeconds} сек...")
                time.sleep(waitSeconds)
                continue

            if response.status_code != 200:
                print(f"  Ошибка друзей HTTP {response.status_code} для {userSteamId}")
                return []

            data = response.json()
            friendsList = data.get("friendslist", {}).get("friends", [])
            return [friend["steamid"] for friend in friendsList]

        except Exception:
            return []

    return []


def computeStatistics(dataset: list) -> dict:
    totalUsers = len(dataset)
    usersWithData = [user for user in dataset if not user.get("error")]
    totalGamesOwned = sum(user.get("gameCount", 0) for user in usersWithData)

    gameOwnershipCounter = {}
    for user in usersWithData:
        for game in user.get("ownedGames", []):
            appId = game.get("appid")
            if appId:
                gameOwnershipCounter[appId] = gameOwnershipCounter.get(appId, 0) + 1

    mostPopularGameName = "Нет данных"
    mostPopularCount = 0
    if gameOwnershipCounter:
        mostPopularAppId = max(gameOwnershipCounter, key=gameOwnershipCounter.get)
        mostPopularCount = gameOwnershipCounter[mostPopularAppId]
        for user in usersWithData:
            for game in user.get("ownedGames", []):
                if game.get("appid") == mostPopularAppId:
                    mostPopularGameName = game.get("name", "Unknown")
                    break
            if mostPopularGameName != "Нет данных":
                break

    averageGamesPerUser = round(totalGamesOwned / totalUsers, 2) if totalUsers > 0 else 0

    return {
        "numberOfUsersProcessed": totalUsers,
        "usersWithSuccessfulData": len(usersWithData),
        "totalGamesOwnedAcrossAllUsers": totalGamesOwned,
        "averageGamesOwnedPerUser": averageGamesPerUser,
        "mostPopularGameName": mostPopularGameName,
        "mostPopularGameOwnershipCount": mostPopularCount
    }

In [ ]:
print("Запускаем сбор датасета по друзьям...")

collectedDataset = collectAccountData(
    steamApiKey=steamApiKey,
    startingUserId=startingSteamId,
    maxUsers=21000,
    maxDepth=15,
    maxWorkersForOwnedGames=5,
    datasetFilePath="../data/steamUsersDataset.json"
)

with open("../data/steamUsersDataset.json", "w", encoding="utf-8") as file:
    json.dump(collectedDataset, file, ensure_ascii=False, indent=2)
    print(f"\nДатасет сохранён в ../data/steamUsersDataset.json ({len(collectedDataset)} пользователей)")

statistics = computeStatistics(collectedDataset)
with open("../data/simpleStatistics.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(statistics, ensure_ascii=False, indent=2))
    print(f"\nСтатистика сохранена в ../data/simpleStatistics.txt")

Запускаем сбор датасета по друзьям...
Найдено 20000 существующих профилей в ../data/steamUsersDataset.json
Нужно собрать ещё 1000 новых пользователей...
Найден новый пользователь 1/1000: 76561198345337589 (глубина 4)...
Найден новый пользователь 2/1000: 76561198345703631 (глубина 4)...
  Ошибка друзей HTTP 401 для 76561198345703631
Найден новый пользователь 3/1000: 76561198348154402 (глубина 4)...
Найден новый пользователь 4/1000: 76561198348315844 (глубина 4)...
Найден новый пользователь 5/1000: 76561198350827568 (глубина 4)...
Найден новый пользователь 6/1000: 76561198354076726 (глубина 4)...
Найден новый пользователь 7/1000: 76561198354775012 (глубина 4)...
Найден новый пользователь 8/1000: 76561198366347572 (глубина 4)...
Найден новый пользователь 9/1000: 76561198371773991 (глубина 4)...
Найден новый пользователь 10/1000: 76561198374028900 (глубина 4)...
Найден новый пользователь 11/1000: 76561198379366398 (глубина 4)...
Найден новый пользователь 12/1000: 76561198383997793 (глубина

### Сбор данных по играм

In [ ]:
def loadExistingGamesDataset(datasetFilePath: str = "../data/steamGamesDataset.json") -> list:
    if os.path.exists(datasetFilePath):
        try:
            with open(datasetFilePath, "r", encoding="utf-8") as file:
                existingGames = json.load(file)
            print(f"Найдено {len(existingGames)} существующих игр в {datasetFilePath}")
            return existingGames
        except Exception as e:
            print(f"Ошибка чтения существующего steamGamesDataset: {e}. Начинаем с нуля.")
    return []


def getExistingAppIds(gamesDataset: list) -> Set[int]:
    existingIds: Set[int] = set()
    for game in gamesDataset:
        appId = game.get("appid")
        if appId:
            existingIds.add(appId)
    return existingIds


def fetchGameDetailsFromSteamSpy(steamAppId: int) -> dict:
    url = "https://steamspy.com/api.php"
    params = {
        "request": "appdetails",
        "appid": steamAppId
    }

    maxRetries = 5
    for attempt in range(maxRetries):
        try:
            response = requests.get(url, params=params, timeout=15)

            if response.status_code in (429, 420):
                waitSeconds = 2 ** attempt + 1
                print(f"  Rate-limit SteamSpy ({response.status_code}) для appid {steamAppId}. Ждём {waitSeconds} сек... (попытка {attempt+1}/{maxRetries})")
                time.sleep(waitSeconds)
                continue

            if response.status_code != 200:
                return {"appid": steamAppId, "error": f"HTTP {response.status_code}"}

            data = response.json()

            return {
                "appid": steamAppId,
                "name": data.get("name"),
                "positive": data.get("positive"),
                "negative": data.get("negative"),
                "owners": data.get("owners"),
                "averageForever": data.get("average_forever"),
                "medianForever": data.get("median_forever"),
                "languages": data.get("languages"),
                "genre": data.get("genre"),
                "tags": data.get("tags", {})
            }

        except requests.exceptions.RequestException as e:
            return {"appid": steamAppId, "error": f"Сетевая ошибка: {str(e)}"}
        except json.JSONDecodeError:
            return {"appid": steamAppId, "error": "Пустой или некорректный JSON от SteamSpy"}
        except Exception as e:
            return {"appid": steamAppId, "error": str(e)}

    return {"appid": steamAppId, "error": f"Не удалось получить данные после {maxRetries} попыток"}


def fetchAllGamesDetailsParallel(
    missingAppIds: List[int],
    maxWorkers: int = 5
) -> list:
    if not missingAppIds:
        return []

    print(f"Запускаем параллельный сбор деталей для {len(missingAppIds)} НЕДОСТАЮЩИХ игр "
          f"в {maxWorkers} потоках...")

    def processSingleGame(appId: int) -> dict:
        print(f"  [appid {appId}] Запрашиваем данные из SteamSpy...")
        gameInfo = fetchGameDetailsFromSteamSpy(appId)
        time.sleep(0.4)
        return gameInfo

    newGames = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=maxWorkers) as executor:
        futures = [executor.submit(processSingleGame, appId) for appId in sorted(missingAppIds)]
        
        for i, future in enumerate(concurrent.futures.as_completed(futures), 1):
            try:
                gameInfo = future.result()
                newGames.append(gameInfo)
                print(f"  [{i}/{len(missingAppIds)}] Завершена игра {gameInfo.get('appid')} - {gameInfo.get('name', 'Unknown')}")
            except Exception as e:
                print(f"  Критическая ошибка в потоке: {e}")

    return newGames

In [ ]:
print("Начинаем сбор информации по всем уникальным играм...")

with open("../data/steamUsersDataset.json", "r", encoding="utf-8") as file:
    usersDataset = json.load(file)

uniqueAppIds: Set[int] = set()
for user in usersDataset:
    for game in user.get("ownedGames", []):
        appId = game.get("appid")
        if appId:
            uniqueAppIds.add(appId)

print(f"Найдено {len(uniqueAppIds)} уникальных игр в steamUsersDataset")

existingGamesDataset = loadExistingGamesDataset()
existingAppIds = getExistingAppIds(existingGamesDataset)

missingAppIds = list(uniqueAppIds - existingAppIds)

if missingAppIds:
    print(f"Нужно дозапросить {len(missingAppIds)} новых игр...")
    newGames = fetchAllGamesDetailsParallel(
        missingAppIds=missingAppIds,
        maxWorkers=8
    )
    
    fullGamesDataset = existingGamesDataset + newGames
    print(f"Добавлено {len(newGames)} новых записей. Всего игр теперь: {len(fullGamesDataset)}")
else:
    print("Все игры уже присутствуют в steamGamesDataset.")
    fullGamesDataset = existingGamesDataset

with open("../data/steamGamesDataset.json", "w", encoding="utf-8") as file:
    json.dump(fullGamesDataset, file, ensure_ascii=False, indent=2)

print(f"\nСохранено {len(fullGamesDataset)} игр в ../data/steamGamesDataset.json "
      f"(из них новых: {len(missingAppIds) if 'missingAppIds' in locals() else 0})")

Начинаем сбор информации по всем уникальным играм...
Найдено 26072 уникальных игр в steamUsersDataset
Найдено 26071 существующих игр в ../data/steamGamesDataset.json
Нужно дозапросить 1 новых игр...
Запускаем параллельный сбор деталей для 1 НЕДОСТАЮЩИХ игр в 8 потоках...
  [appid 1783540] Запрашиваем данные из SteamSpy...
  [1/1] Завершена игра 1783540 - Movavi Video Suite 2022 Steam Edition - Video Making Software: Video Editor Plus, Screen Recorder and Video Converter Premium
Добавлено 1 новых записей. Всего игр теперь: 26072

Сохранено 26072 игр в ../data/steamGamesDataset.json (из них новых: 1)


### Проверка steamUsersDataset.json + статистика

In [ ]:
usersPath = "../data/steamUsersDataset.json"

if os.path.exists(usersPath):
    with open(usersPath, "r", encoding="utf-8") as file:
        users = json.load(file)
    
    print(f"Пользователи: всего записей = {len(users)}")
    
    successful = [user for user in users if not user.get("error")]
    print(f"Успешных (без error) = {len(successful)}")
    print(f"С ошибками = {len(users) - len(successful)}")
    
    steamIds = [user.get("steamId") for user in users if user.get("steamId")]
    uniqueIds = len(set(steamIds))
    print(f"Дубликатов по steamId = {len(steamIds) - uniqueIds}")
    
    missingName = sum(1 for user in users if not user.get("personaname"))
    missingGames = sum(1 for user in users if not user.get("ownedGames") or len(user.get("ownedGames", [])) == 0)
    missingCountry = sum(1 for user in users if not user.get("loccountrycode"))
    print(f"Пропущено personaname = {missingName}")
    print(f"Пропущено ownedGames = {missingGames}")
    print(f"Пропущено loccountrycode = {missingCountry}")
    
    print(f"Среднее количество игр у успешных: {sum(user.get('gameCount', 0) for user in successful) / len(successful):.1f}" if successful else "Нет успешных")
else:
    print("Файл пользователей не найден")

Пользователи: всего записей = 20000
Успешных (без error) = 9978
С ошибками = 10022
Дубликатов по steamId = 0
Пропущено personaname = 2
Пропущено ownedGames = 10022
Пропущено loccountrycode = 8921
Среднее количество игр у успешных: 82.8


In [ ]:
statsPath = "../data/simpleStatistics.txt"

if os.path.exists(statsPath):
    with open(statsPath, "r", encoding="utf-8") as file:
        print("\nСохранённая статистика:")
        print(file.read())


Сохранённая статистика:
{
  "numberOfUsersProcessed": 20000,
  "usersWithSuccessfulData": 9978,
  "totalGamesOwnedAcrossAllUsers": 825977,
  "averageGamesOwnedPerUser": 41.3,
  "mostPopularGameName": "Counter-Strike 2",
  "mostPopularGameOwnershipCount": 9505
}


### Проверка steamGamesDataset.json

In [ ]:
gamesPath = "../data/steamGamesDataset.json"

if os.path.exists(gamesPath):
    with open(gamesPath, "r", encoding="utf-8") as file:
        games = json.load(file)
    
    print(f"\nИгры: всего записей = {len(games)}")
    
    successfulGames = [game for game in games if not game.get("error")]
    print(f"Успешных игр = {len(successfulGames)}")
    print(f"С ошибками = {len(games) - len(successfulGames)}")
    
    appids = [game.get("appid") for game in games if game.get("appid") is not None]
    uniqueAppIds = len(set(appids))
    print(f"Дубликатов по appid = {len(appids) - uniqueAppIds}")
    
    missingNames = sum(1 for game in games if not game.get("name"))
    missingOwners = sum(1 for game in games if not game.get("owners"))
    print(f"Пропущено name = {missingNames}")
    print(f"Пропущено owners = {missingOwners}")
    
    print(f"Средний playtime (average_forever) среди успешных: {sum(game.get('averageForever', 0) for game in successfulGames) / len(successfulGames):.1f}" if successfulGames else "Нет данных")
else:
    print("Файл игр не найден")


Игры: всего записей = 26072
Успешных игр = 26072
С ошибками = 0
Дубликатов по appid = 0
Пропущено name = 278
Пропущено owners = 0
Средний playtime (average_forever) среди успешных: 1107.6
